# CO3 最小原理教程

这个 notebook 用 `minimal-co3/` 里的 Python demo 解释 CO3 的整体原理。

重点不是复现完整论文系统，而是看懂这条主线：

```text
固件在 MCU 上具体执行
    -> 插桩只上报和符号推理有关的运行时事实
    -> 主机根据编译期提取的 SVFG 还原约束
    -> Z3 求出新输入
    -> 下一轮设备跑得更深
```

先记住一句话：**CO3 把“真实执行”留在设备端，把“符号推理”放到主机端。**

## 1. CO3 要解决什么问题

传统固件符号执行经常卡在两个地方：

1. **硬件外设不好模拟**：MMIO、中断、DMA 等行为很难在主机上完整复现。
2. **调试接口太慢**：如果每次硬件访问都通过 GDB/JTAG 同步，速度很差。

CO3 的选择是：

```text
MCU/device:       运行真实固件，只发送必要 trace
workstation/host: 保存 SVFG，构造符号约束，调用 Z3
```

所以设备端不是求解器，设备端只是“具体执行 + 上报少量事实”。

## 2. CO3 的两个阶段

### 编译期/offline

CO3 编译器从固件源码/LLVM IR 中提取：

```text
bin*          带插桩的固件，跑在 MCU 上
SVFG          Symbolic Value Flow Graph，保存在主机上
```

SVFG 记录的是：符号输入如何经过函数参数、计算、分支，最终变成约束。

### 运行期/online

```text
MCU:  execute(bin*, input) -> trace
Host: build(SVFG, trace) -> constraints -> solve -> new input
```

这个 demo 用 `common.py` 模拟“编译期提取的 SVFG”，用 `device.py` 模拟 MCU，用 `host.py` 模拟 workstation。

## 3. 本 demo 的文件对应关系

| 文件 | 对应 CO3 概念 |
|---|---|
| `device.py` | MCU/固件端：具体执行协议逻辑，输出 CO3 trace |
| `common.py` | 编译期信息：插桩 ID、函数 ID、分支 ID、`EXTRACTED_SVFG` |
| `host.py` | workstation：解析 trace，绑定 SVFG，调用 Z3 |
| `co3_trace.txt` | 设备端最近一轮输出的原始 trace |
| `co3_log.txt` | 主机端解释 trace、约束和 Z3 model 的详细日志 |

最小闭环会跑三轮：

```text
round 1: 地址分支失败 -> 求 input_0 = 0x5a
round 2: 长度分支失败 -> 求 input_1 in [1, 4]
round 3: 两个分支都通过 -> BUG path reached
```

In [9]:
from pathlib import Path
import sys

# 让 notebook 无论从仓库根目录还是 minimal-co3 目录启动，都能 import demo 模块。
CWD = Path.cwd()
DEMO_DIR = CWD if (CWD / "common.py").exists() else CWD / "minimal-co3"
if str(DEMO_DIR) not in sys.path:
    sys.path.insert(0, str(DEMO_DIR))

# print("DEMO_DIR =", DEMO_DIR)

## 4. 设备端：只做具体执行 + 插桩上报

设备端协议逻辑非常简单：

```python
addr = sym_read(0)
length = sym_read(1)

addr_ok = check_addr(addr)
if not addr_ok: return False

len_ok = check_length(length)
if not len_ok: return False

BUG
```

设备端上报的事件类型定义在 `common.py`：

```text
MEM_R   符号内存读
CALL    函数调用，告诉主机 param0 来自哪个 input
RET     函数返回 true/false
BRANCH  分支结果，pass-to-solver 节点
BUG     目标路径
```

注意：设备端不会调用 Z3，也不会知道完整 SVFG。

In [10]:
import common

print("插桩类型:")
for name in ["MEM_R", "CALL", "RET", "BRANCH", "BUG"]:
    print(f"  {name} =", getattr(common, name))

print("\n分支 ID:")
print("  BRANCH_ADDR =", common.BRANCH_ADDR)
print("  BRANCH_LEN  =", common.BRANCH_LEN)

print("\n函数 ID:")
print("  FUNC_CHECK_ADDR   =", common.FUNC_CHECK_ADDR)
print("  FUNC_CHECK_LENGTH =", common.FUNC_CHECK_LENGTH)

插桩类型:
  MEM_R = 1
  CALL = 4
  RET = 5
  BRANCH = 3
  BUG = 9

分支 ID:
  BRANCH_ADDR = 1
  BRANCH_LEN  = 2

函数 ID:
  FUNC_CHECK_ADDR   = 1
  FUNC_CHECK_LENGTH = 2


## 5. 为什么要有 CALL/RET

如果只有分支结果：

```text
BRANCH_LEN = false
```

主机只知道“长度分支失败”，但不知道分支函数里的 `param0` 本轮来自哪个输入。

本 demo 让 `CALL` 记录参数绑定：

```text
CALL check_length(param0 <- input[1])
```

这对应 CO3 论文里“SVFG 可通过 function call 连接”的意思：

```text
MEM_R input[1]
  -> CALL check_length(param0 <- input[1])
  -> RET check_length
  -> BRANCH_LEN
```

所以 CALL/RET 的作用是：把跨函数的符号流讲清楚。

## 6. 编译期提取信息：EXTRACTED_SVFG

真实 CO3 会从 LLVM IR 里提取 SVFG。这个 demo 用 `common.EXTRACTED_SVFG` 手写模拟：

```python
EXTRACTED_SVFG = {
    FUNC_CHECK_ADDR:   ("eq_const", "param0", 0x5A),
    FUNC_CHECK_LENGTH: ("range_u8", "param0", 1, 4),
}
```

它的意思是：

```text
check_addr.param0 == 0x5A
1 <= check_length.param0 <= 4
```

注意这里写的是 `param0`，不是 `input_0` 或 `input_1`。实际 `param0` 绑定到哪个输入，要等运行时 `CALL` trace 告诉主机。

### 6.1 SVFG 是什么

SVFG 全称是 **Symbolic Value-Flow Graph**，可以理解成“符号值流图”。它回答的问题不是“这次运行的值是多少”，而是：

```text
一个符号输入从哪里来？
经过了哪些赋值、函数参数、返回值？
最后影响了哪个分支条件？
这个分支条件的表达式长什么样？
```

在 CO3 里，SVFG 是编译期从固件中提取出来的静态信息。它不是运行时 trace，也不是 Z3 求解结果，而是主机生成约束时用的“结构模板”。

用本 demo 的源码举例：

```python
def check_length(length, input_offset):
    result = (1 <= length <= 4)
    return result
```

编译期 SVFG 提取到的重点不是当前 `length` 等于多少，而是：

```text
check_length 的返回值由 param0 决定
返回值为 true 的条件是 1 <= param0 <= 4
BRANCH_LEN 使用 check_length 的返回值作为分支条件
```

所以在 `common.py` 里我们用两张小表模拟真实 CO3 的 SVFG 信息：

```python
BRANCH_TO_FUNC = {
    BRANCH_ADDR: FUNC_CHECK_ADDR,
    BRANCH_LEN:  FUNC_CHECK_LENGTH,
}

EXTRACTED_SVFG = {
    FUNC_CHECK_ADDR:   ("eq_const", "param0", 0x5A),
    FUNC_CHECK_LENGTH: ("range_u8", "param0", 1, 4),
}
```

这里仍然只有 `param0`，还不知道它对应协议输入的第几个字节。这个信息要等设备运行时通过 `CALL` 事件告诉主机：

```text
CALL check_length(param0 <- input[1])
```

于是主机才能完成绑定：

```text
SVFG 模板:    1 <= check_length.param0 <= 4
CALL 绑定:    check_length.param0 <- input_1
Z3 约束:      1 <= input_1 <= 4
```

这就是为什么本 demo 里既要有 `EXTRACTED_SVFG`，也要有 `CALL/RET` trace：SVFG 给出“函数内部逻辑”，CALL 给出“这次运行时实参来自哪个输入”。两者合起来，主机才知道该修改哪个输入字节。


In [11]:
from pprint import pprint

print("BRANCH_TO_FUNC:")
pprint(common.BRANCH_TO_FUNC)

print("\nEXTRACTED_SVFG:")
pprint(common.EXTRACTED_SVFG)

BRANCH_TO_FUNC:
{1: 1, 2: 2}

EXTRACTED_SVFG:
{1: ('eq_const', 'param0', 90), 2: ('range_u8', 'param0', 1, 4)}


## 7. 主机端如何把 trace + SVFG 变成约束

主机做四件事：

1. 读 `co3_trace.txt`
2. 找出 `MEM_R`，创建 Z3 符号变量 `input_0`、`input_1`
3. 找出 `CALL`，得到函数参数绑定：例如 `check_length.param0 <- input_1`
4. 找出第一个失败 `BRANCH`，用 SVFG 生成“让它变 true”的约束

关键点：**翻转不是修改设备代码，而是求出下一轮输入，让同一个分支在下一轮自然变成 true。**

In [12]:
import host

# 看看 host 如何把函数级 SVFG 绑定成具体 Z3 公式。
from z3 import BitVec
sym_vars = {0: BitVec("input_0", 8), 1: BitVec("input_1", 8)}

# 假设 trace 里有：CALL check_addr(param0 <- input[0])
call_args = {common.FUNC_CHECK_ADDR: 0}
print(host.build_formula_from_extracted_svfg(common.BRANCH_ADDR, sym_vars, call_args))

# 假设 trace 里有：CALL check_length(param0 <- input[1])
call_args = {common.FUNC_CHECK_LENGTH: 1}
print(host.build_formula_from_extracted_svfg(common.BRANCH_LEN, sym_vars, call_args))

input_0 == 90
And(UGE(input_1, 1), ULE(input_1, 4))


## 8. 运行完整最小闭环

下面直接运行 `host.py`。它会：

```text
初始输入 {0: 0x10, 1: 0x09}
  -> 第 1 轮 addr 失败，求 input_0 = 0x5a
  -> 第 2 轮 len 失败，求 input_1 = 0x04
  -> 第 3 轮触发 BUG
```

In [13]:
import subprocess

result = subprocess.run(
    [sys.executable, str(DEMO_DIR / "host.py")],
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)


round 1, input = {0: 0x10, 1: 0x09}
[device] run firmware
[device] reject: bad addr
[host] trace records: 5
[host] symbolic inputs: [0, 1]
[host] call args: {1: 0}
[host] branches: [(1, False)]
[host] flip BRANCH_ADDR: false -> true
[host] next input: {0: 0x5a, 1: 0x09}

round 2, input = {0: 0x5a, 1: 0x09}
[device] run firmware
[device] reject: bad len
[host] trace records: 8
[host] symbolic inputs: [0, 1]
[host] call args: {1: 0, 2: 1}
[host] branches: [(1, True), (2, False)]
[host] keep BRANCH_ADDR=true
[host] flip BRANCH_LEN: false -> true
[host] next input: {0: 0x5a, 1: 0x04}

round 3, input = {0: 0x5a, 1: 0x04}
[device] run firmware
[device] BUG path reached
[host] done
[host] z3 log saved to /Users/fripside/Work/Dev/Core/Research/RTOS/IoTSymDemo/co3-demo/minimal-co3/co3_log.txt



## 9. 查看设备原始 trace

`co3_trace.txt` 是设备端发送给主机的“线上数据”。它很小，基本只有数字：

```text
stub_type tag value
```

主机靠 `common.py` 里的 ID 和 SVFG 信息把它解释成人能读懂的符号流。

In [14]:
print((DEMO_DIR / "co3_trace.txt").read_text(encoding="utf-8"))

1 0 90
1 1 4
4 1 0
5 1 1
3 1 1
4 2 1
5 2 1
3 2 1
9 0 1



## 10. 查看主机详细 log

`co3_log.txt` 更适合学习，它会把 trace 解码，并展示 Z3 里到底有什么：

- `decoded trace`
- `call bindings`
- `constraints added to Z3`
- `solver.sexpr()`
- `model`
- `next input`

In [15]:
log_text = (DEMO_DIR / "co3_log.txt").read_text(encoding="utf-8")
print(log_text)

round 1
device input: {0: 0x10, 1: 0x09}
decoded trace:
  MEM_R    input[0] = 0x10 (16)    raw=(1, 0, 16)
  MEM_R    input[1] = 0x09 (9)    raw=(1, 1, 9)
  CALL     check_addr(param0 <- input[0])    raw=(4, 1, 0)
  RET      check_addr -> false    raw=(5, 1, 0)
  BRANCH   BRANCH_ADDR -> false    raw=(3, 1, 0)
symbolic inputs:
  input_0: concrete=0x10, z3=input_0
  input_1: concrete=0x09, z3=input_1
call bindings:
  check_addr.param0 <- input_0, return=false
branches:
  BRANCH_ADDR: false
constraints added to Z3:
SVFG bind: BRANCH_ADDR uses check_addr.param0 <- input_0
  flip BRANCH_ADDR: false -> true: input_0 == 90
solver.sexpr():
(declare-fun input_0 () (_ BitVec 8))
(assert (= input_0 #x5a))

solver.check(): sat
model:
  input_0 = 90
current input: {0: 0x10, 1: 0x09}
next input:    {0: 0x5a, 1: 0x09}
round 2
device input: {0: 0x5a, 1: 0x09}
decoded trace:
  MEM_R    input[0] = 0x5a (90)    raw=(1, 0, 90)
  MEM_R    input[1] = 0x09 (9)    raw=(1, 1, 9)
  CALL     check_addr(param0 <- 

## 11. 手动复现第二轮 Z3 约束

第二轮中，主机保留已经通过的地址分支，再翻转长度分支：

```text
keep BRANCH_ADDR=true: input_0 == 0x5a
flip BRANCH_LEN=false->true: 1 <= input_1 <= 4
```

这就是 concolic execution 常见的路径探索方式：

```text
保留路径前缀 + 翻转下一个失败分支
```

In [16]:
from z3 import BitVec, Solver, UGE, ULE, And, sat

input_0 = BitVec("input_0", 8)
input_1 = BitVec("input_1", 8)

s = Solver()
s.add(input_0 == 0x5A)                 # 保留已通过地址分支
s.add(And(UGE(input_1, 1), ULE(input_1, 4)))  # 翻转长度分支为 true

print(s.sexpr())
print(s.check())
print(s.model())

(declare-fun input_0 () (_ BitVec 8))
(declare-fun input_1 () (_ BitVec 8))
(assert (= input_0 #x5a))
(assert (and (bvuge input_1 #x01) (bvule input_1 #x04)))

sat
[input_1 = 4, input_0 = 90]


## 12. 这个 demo 和真实 CO3 的差异

这个 demo 只保留最核心概念：

```text
设备具体执行 -> 上报 trace -> 主机绑定 SVFG -> Z3 求新输入
```

为了方便理解，它没有实现真实 CO3 的很多工程细节：

- 没有 LLVM 自动提取 SVFG，`EXTRACTED_SVFG` 是手写的
- 没有真实 MCU/UART，设备和主机都跑在 Python 里
- 没有完整 shadow memory，只演示 `MEM_R` 和函数参数绑定
- 没有 fuzzing 调度，只做固定三轮路径深入

但理解这个最小闭环后，再看 C 版 `src-demo/` 和 CO3 论文会顺很多。